# Decision Tree

A Decision Tree is one of the most intuitive supervised learning algorithms. It builds a tree-like structure where each internal node represents a decision based on a feature, each branch represents the outcome of that decision, and each leaf node holds a predicted value. Decision Trees handle both **regression** tasks — where the leaf returns the mean of continuous target values — and **classification** tasks — where the leaf returns the most frequent class label. In both cases, the tree learns to split the data recursively to minimize a task-specific impurity measure.

## 1. Key Concepts

A Decision Tree is a **non-parametric, eager learner** — unlike KNN, it builds an explicit model during training. The tree is constructed by recursively partitioning the training data $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$ into subsets that are increasingly homogeneous with respect to the target variable $y$.

At each node, the algorithm selects the best **(feature, threshold)** pair that maximizes the reduction of impurity. The process continues until a stopping criterion is met, at which point the node becomes a **leaf**. The leaf value depends on the task:

| Task | Leaf value | Formula |
|------|-----------|--------|
| **Regression** | Mean of target values | $\hat{y} = \dfrac{1}{\|\mathcal{S}\|} \displaystyle\sum_{i \in \mathcal{S}} y_i$ |
| **Classification** | Most frequent class label | $\hat{y} = \arg\max_{c} \displaystyle\sum_{i \in \mathcal{S}} \mathbf{1}[y_i = c]$ |

Where $\mathcal{S}$ is the set of sample indices at the leaf node.

## 2. Splitting Criteria

The splitting criterion is the core difference between regression and classification trees. Both aim to find the **(feature, threshold)** pair that maximizes **impurity reduction** $\Delta$, but they measure impurity differently.

### 2.1 Regression — Variance Reduction (MSE)

For regression, node impurity is measured by the **Mean Squared Error** of the target values:

$$\text{MSE}(\mathcal{S}) = \frac{1}{|\mathcal{S}|} \sum_{i \in \mathcal{S}} (y_i - \bar{y})^2$$

The **variance reduction** for a split into $\mathcal{S}_L$ and $\mathcal{S}_R$ is:

$$\Delta_{\text{reg}} = \text{MSE}(\mathcal{S}) - \left( \frac{|\mathcal{S}_L|}{|\mathcal{S}|} \cdot \text{MSE}(\mathcal{S}_L) + \frac{|\mathcal{S}_R|}{|\mathcal{S}|} \cdot \text{MSE}(\mathcal{S}_R) \right)$$

### 2.2 Classification — Information Gain (Entropy)

For classification, node impurity is commonly measured by **Entropy**, which quantifies the disorder of class labels in a node:

$$H(\mathcal{S}) = - \sum_{c} p_c \log_2(p_c)$$

Where $p_c = \frac{|\{i \in \mathcal{S} : y_i = c\}|}{|\mathcal{S}|}$ is the proportion of class $c$ in the node. A pure node (all one class) has $H = 0$; maximum disorder gives $H = \log_2(K)$ for $K$ classes.

The **information gain** for a split is:

$$\Delta_{\text{cls}} = H(\mathcal{S}) - \left( \frac{|\mathcal{S}_L|}{|\mathcal{S}|} \cdot H(\mathcal{S}_L) + \frac{|\mathcal{S}_R|}{|\mathcal{S}|} \cdot H(\mathcal{S}_R) \right)$$

An alternative to Entropy is the **Gini Impurity**, which is computationally cheaper (no logarithm) and often gives similar results:

$$\text{Gini}(\mathcal{S}) = 1 - \sum_{c} p_c^2$$

### 2.3 Comparison at a Glance

| | Regression | Classification |
|-|-----------|----------------|
| **Impurity measure** | MSE (variance) | Entropy or Gini impurity |
| **Gain metric** | Variance Reduction | Information Gain |
| **Pure node** | All $y_i$ identical → MSE = 0 | All $y_i$ same class → $H$ = 0 or Gini = 0 |
| **Leaf prediction** | $\bar{y}$ (mean) | $\arg\max_c$ (majority class) |

**Note**: The implementation uses a candidate threshold strategy for efficiency. If a feature has more than 10 unique values, only the 25th, 50th, and 75th percentiles are evaluated as candidate thresholds, reducing computation without significantly sacrificing accuracy.

## 3. Stopping Criteria

The tree stops growing a branch (i.e., creates a leaf node) when **any** of the following conditions is met. These criteria apply identically to both regression and classification:

| Condition | Parameter | Description |
|-----------|-----------|-------------|
| Too few samples to split | `min_samples_split` | Node has fewer samples than the threshold |
| Maximum depth reached | `max_depth` | Tree has reached the allowed depth |
| Pure node | — | All target values in the node are identical |
| Insufficient gain | `min_impurity_decrease` | No split achieves the required impurity reduction |
| Leaf too small | `min_samples_leaf` | A split would produce a child with too few samples |

## 4. Pseudo-algorithm

The tree-building logic is identical for both tasks — only the impurity function $\mathcal{I}$ and the leaf value $\ell$ change.

**Build phase** — `fit(X, y, depth)`:

1. $\mathcal{D} \leftarrow$ training set of $N$ samples $(\mathbf{x}_i, y_i)$
2. **if** stopping criterion met **then return** $\text{leaf}(\ell)$ where:
   - Regression: $\ell = \bar{y}$
   - Classification: $\ell = \arg\max_c\, |\{y_i = c\}|$
3. **for each** feature $j$ and candidate threshold $t$ **do**
4. $\quad \mathcal{S}_L \leftarrow \{i : x_{ij} < t\}$, $\mathcal{S}_R \leftarrow \{i : x_{ij} \geq t\}$
5. $\quad \Delta_{j,t} \leftarrow \mathcal{I}(\mathcal{S}) - \tfrac{|\mathcal{S}_L|}{|\mathcal{S}|}\mathcal{I}(\mathcal{S}_L) - \tfrac{|\mathcal{S}_R|}{|\mathcal{S}|}\mathcal{I}(\mathcal{S}_R)$ &nbsp;&nbsp; *(MSE or Entropy)*
6. **end for**
7. $(j^*, t^*) \leftarrow \arg\max_{j,t} \; \Delta_{j,t}$
8. **if** $\Delta_{j^*, t^*} <$ `min_impurity_decrease` **then return** $\text{leaf}(\ell)$
9. **return** Node$(j^*, t^*,$ left=`fit`$(\mathcal{S}_L,$ depth$+1)$, right=`fit`$(\mathcal{S}_R,$ depth$+1))$

**Predict phase** — `predict(x)`:

10. **if** node is a leaf **then return** $\hat{y}$ (leaf value)
11. **if** $x[j^*] < t^*$ **then** traverse left child
12. **else** traverse right child
13. **return** prediction at reached leaf

## 5. Implementation

We demonstrate both tasks side by side:
- **Regression** uses the **California Housing** dataset (20,640 samples, 8 features, continuous target: median house value).
- **Classification** uses the **Iris** dataset (150 samples, 4 features, 3 classes: Setosa, Versicolor, Virginica).

In [ ]:
import pandas as pd
import time
from sklearn.datasets import fetch_california_housing, load_iris
from ifri_mini_ml_lib.preprocessing.preparation import DataSplitter

# --- Regression dataset ---
housing = fetch_california_housing()
X_reg = pd.DataFrame(housing.data, columns=housing.feature_names)
y_reg = pd.Series(housing.target)

# --- Classification dataset ---
iris = load_iris()
X_clf = pd.DataFrame(iris.data, columns=iris.feature_names)
y_clf = pd.Series(iris.target)

splitter = DataSplitter(seed=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = splitter.train_test_split(X_reg, y_reg, test_size=0.2)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = splitter.train_test_split(X_clf, y_clf, test_size=0.2)

### 5.1 Regression

#### With Scikit-learn

In [ ]:
from sklearn.tree import DecisionTreeRegressor as SklearnDTR

start_1 = time.perf_counter()
sk_tree_reg = SklearnDTR(max_depth=5, min_samples_split=2, min_samples_leaf=1)
sk_tree_reg.fit(X_train_reg, y_train_reg)
end_1 = time.perf_counter()

#### With ifri-mini-ml-lib

In [ ]:
from ifri_mini_ml_lib.regression import DecisionTreeRegressor

start_2 = time.perf_counter()
my_tree_reg = DecisionTreeRegressor(max_depth=5, min_samples_split=2, min_samples_leaf=1)
my_tree_reg.fit(X_train_reg.values, y_train_reg.values)
end_2 = time.perf_counter()

y_pred_reg_1 = sk_tree_reg.predict(X_test_reg)
y_pred_reg_2 = my_tree_reg.predict(X_test_reg.values)

In [ ]:
from ifri_mini_ml_lib.metrics.regression import mean_squared_error, mean_absolute_error, r2_score

reg_results = pd.DataFrame({
    'Metric': ['MSE', 'MAE', 'R² Score', 'Training Time (s)'],
    'Scikit-learn': [
        mean_squared_error(y_test_reg, y_pred_reg_1),
        mean_absolute_error(y_test_reg, y_pred_reg_1),
        r2_score(y_test_reg, y_pred_reg_1),
        end_1 - start_1
    ],
    'ifri-mini-ml-lib': [
        mean_squared_error(y_test_reg, y_pred_reg_2),
        mean_absolute_error(y_test_reg, y_pred_reg_2),
        r2_score(y_test_reg, y_pred_reg_2),
        end_2 - start_2
    ],
})
reg_results.T

### 5.2 Classification

#### With Scikit-learn

In [ ]:
from sklearn.tree import DecisionTreeClassifier as SklearnDTC

start_3 = time.perf_counter()
sk_tree_clf = SklearnDTC(criterion='entropy', max_depth=5, min_samples_split=2, min_samples_leaf=1)
sk_tree_clf.fit(X_train_clf, y_train_clf)
end_3 = time.perf_counter()

#### With ifri-mini-ml-lib

In [ ]:
from ifri_mini_ml_lib.classification import DecisionTreeClassifier

start_4 = time.perf_counter()
my_tree_clf = DecisionTreeClassifier(criterion='entropy', max_depth=5, min_samples_split=2, min_samples_leaf=1)
my_tree_clf.fit(X_train_clf.values, y_train_clf.values)
end_4 = time.perf_counter()

y_pred_clf_1 = sk_tree_clf.predict(X_test_clf)
y_pred_clf_2 = my_tree_clf.predict(X_test_clf.values)

In [ ]:
from ifri_mini_ml_lib.metrics.classification import accuracy_score, precision_score, recall_score, f1_score

clf_results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Training Time (s)'],
    'Scikit-learn': [
        accuracy_score(y_test_clf, y_pred_clf_1),
        precision_score(y_test_clf, y_pred_clf_1, average='macro'),
        recall_score(y_test_clf, y_pred_clf_1, average='macro'),
        f1_score(y_test_clf, y_pred_clf_1, average='macro'),
        end_3 - start_3
    ],
    'ifri-mini-ml-lib': [
        accuracy_score(y_test_clf, y_pred_clf_2),
        precision_score(y_test_clf, y_pred_clf_2, average='macro'),
        recall_score(y_test_clf, y_pred_clf_2, average='macro'),
        f1_score(y_test_clf, y_pred_clf_2, average='macro'),
        end_4 - start_4
    ],
})
clf_results.T

## 6. Interactive Demo

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown
from notebooks.utils import plot_decision_tree

interact(
    plot_decision_tree,
    task=Dropdown(
        options=['regression', 'classification'],
        value='regression',
        description='Task:'
    ),
    criterion=Dropdown(
        options=['mse', 'entropy', 'gini'],
        value='mse',
        description='Criterion:'
    ),
    max_depth=IntSlider(min=1, max=10, step=1, value=3, description='Max Depth:'),
    min_samples_split=IntSlider(min=2, max=20, step=1, value=2, description='Min Split:'),
    min_samples_leaf=IntSlider(min=1, max=20, step=1, value=1, description='Min Leaf:'),
    library=Dropdown(
        options=['sklearn', 'ifri_mini_ml_lib'],
        value='sklearn',
        description='Library:'
    )
);

The interactive demo lets you switch between tasks and criteria to observe their influence on the tree's behavior. In **regression** mode, switching from a shallow to a deep tree shows the transition from underfitting to overfitting. In **classification** mode, comparing `entropy` vs `gini` typically yields similar decision boundaries but may differ in edge cases with small class imbalances. In both tasks, increasing `min_samples_split` and `min_samples_leaf` constrains tree growth and acts as a regularization mechanism.

## 7. Real-life Applications

Decision Trees are widely adopted across industries thanks to their interpretability and ability to handle both regression and classification tasks without feature scaling.

1. **Medical Diagnosis (Classification)**: Decision trees are used to classify patients into risk categories (e.g., diabetic / non-diabetic, benign / malignant tumor) based on clinical measurements. Their transparent rules make them auditable and explainable to healthcare professionals.

2. **Real Estate Pricing (Regression)**: Trees estimate property values from location, size, and amenity features. The interpretable split structure makes it easy to justify pricing decisions to stakeholders.

3. **Credit Scoring (Classification)**: Banks use decision trees to classify loan applicants as creditworthy or not based on income, debt ratio, and repayment history — with the added benefit that the decision path can be disclosed to regulators.

4. **Energy Forecasting (Regression)**: Utilities predict electricity consumption based on weather, time, and historical patterns. Trees handle the non-linear seasonal relationships without requiring feature transformation.

Decision trees also serve as the **base learner** in powerful ensemble methods such as **Random Forests** and **Gradient Boosting Machines**, which apply to both regression and classification problems.

## 8. Limitations and Challenges

Despite their intuitive appeal, decision trees have notable weaknesses that apply to both regression and classification settings.

1. **Overfitting**: Without constraints, a decision tree can grow deep enough to memorize the training set entirely — achieving near-zero training error while performing poorly on unseen data. Regularization through `max_depth`, `min_samples_leaf`, and `min_impurity_decrease` is essential in both tasks.

2. **High Variance (Instability)**: Small changes in the training data can lead to very different tree structures. A single tree is highly sensitive to noise and outliers, making it an unstable predictor regardless of the task.

3. **Axis-Aligned Splits Only**: The splitting criterion always partitions the feature space with axis-parallel hyperplanes. Diagonal or curved decision boundaries (common in classification) require a large number of splits to approximate, increasing model complexity.

4. **Class Imbalance Sensitivity (Classification)**: In classification tasks, a majority class can dominate the splits, causing the tree to poorly learn the minority class. Techniques such as class weighting or resampling are needed to address this.

These limitations motivate the use of ensemble methods (Random Forest, Gradient Boosting) that combine many trees to reduce variance and improve generalization across both regression and classification problems.

## 9. References

- Decision Trees, Scikit-learn User Guide, https://scikit-learn.org/stable/modules/tree.html
- Breiman, L. et al. (1984). *Classification and Regression Trees (CART)*. Wadsworth.
- Quinlan, J. R. (1986). Induction of Decision Trees. *Machine Learning*, 1(1), 81–106. *(ID3 algorithm — origin of Information Gain)*
- Decision Trees, StatQuest with Josh Starmer, https://www.youtube.com/watch?v=_L39rN6gz7Y